# Auto-Labeling dengan Distribusi Target

1. Load & train LR model pada `comments_sentiment`
2. Prediksi label pada `comments_preprocessed` (exclude data yang sudah ada di `comments_sentiment`)
3. Stratified sampling: positif=30%, negatif=40%, netral=30% (max 2000)
4. Simpan ke MongoDB dengan kolom: `comment_id, video_id, text_original, text_final, sentiment`


## 1. Import & Konfigurasi


In [ ]:
import csv
import os
from pathlib import Path
from datetime import datetime, timezone

from dotenv import load_dotenv
from pymongo import MongoClient
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import (
    CountVectorizer, IDF, NGram, RegexTokenizer,
    StringIndexer, VectorAssembler,
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import FloatType


In [ ]:
TEXT_COL = "text_final"
LABEL_COL = "sentiment"
VALID_LABELS = ("positif", "netral", "negatif")
SEED = 42
TARGET_TOTAL = 2000

# Distribusi target: 30% positif, 40% negatif, 30% netral
TARGET_DIST = {
    "positif": 0.30,
    "negatif": 0.40,
    "netral": 0.30,
}
TARGET_COUNTS = {lbl: int(round(TARGET_TOTAL * p)) for lbl, p in TARGET_DIST.items()}
diff = TARGET_TOTAL - sum(TARGET_COUNTS.values())
if diff != 0:
    lbl = "negatif"
    TARGET_COUNTS[lbl] = TARGET_COUNTS[lbl] + diff

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(dotenv_path=PROJECT_ROOT / ".env", override=True)
MONGO_URI = os.getenv("MONGO_URI", "").strip()
MONGO_DB = os.getenv("MONGO_DB", "analisis_sentimen").strip()
MONGO_LABELED_COLLECTION = "comments_sentiment"
MONGO_PREPROC_COLLECTION = "comments_preprocessed"
MONGO_OUTPUT_COLLECTION = "comments_auto_balanced"

print(f"Distribusi target ({TARGET_TOTAL} baris):")
for lbl, cnt in TARGET_COUNTS.items():
    print(f"  {lbl}: {cnt} ({TARGET_DIST[lbl]*100:.0f}%)")
print(f"Output: {MONGO_OUTPUT_COLLECTION}")


In [ ]:
def create_spark_session(app_name: str = "auto-label-distribusi") -> SparkSession:
    java_home = os.environ.get("JAVA_HOME") or r"C:\\Program Files\\Java\\jdk-22"
    os.environ["JAVA_HOME"] = java_home
    os.environ["PYSPARK_PYTHON"] = os.environ.get("PYSPARK_PYTHON") or "python"
    spark = SparkSession.builder \
        .appName(app_name) \
        .master("local[*]") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.driver.memory", "8g") \
        .config("spark.sql.shuffle.partitions", "8") \
        .config("spark.default.parallelism", "8") \
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
        .getOrCreate()
    spark._jsc.hadoopConfiguration().set(
        "fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem"
    )
    return spark


def _normalize_value(v):
    if v is None:
        return ''
    if isinstance(v, (int, float)):
        return str(v)
    return str(v)


def build_pipeline(**kwargs):
    tokenizer = RegexTokenizer(
        inputCol=TEXT_COL, outputCol="tokens",
        pattern=r'\s+', gaps=True, minTokenLength=2,
    )
    label_indexer = StringIndexer(
        inputCol=LABEL_COL, outputCol="label_index", handleInvalid="keep"
    )
    ngram = NGram(n=2, inputCol="tokens", outputCol="bigrams")
    vocab_uni = kwargs.get("vocab_uni", 8000)
    vocab_bi = kwargs.get("vocab_bi", 6000)
    min_df = kwargs.get("min_df", 3.0)
    cv_uni = CountVectorizer(
        inputCol="tokens", outputCol="uni_feat",
        vocabSize=vocab_uni, minDF=min_df, minTF=1,
    )
    cv_bi = CountVectorizer(
        inputCol="bigrams", outputCol="bi_feat",
        vocabSize=vocab_bi, minDF=min_df, minTF=1,
    )
    assembler = VectorAssembler(
        inputCols=["uni_feat", "bi_feat"], outputCol="raw_feat"
    )
    idf = IDF(inputCol="raw_feat", outputCol="features", minDocFreq=2)
    clf = LogisticRegression(
        featuresCol="features", labelCol="label_index", predictionCol="pred_index",
        maxIter=kwargs.get("max_iter", 300), regParam=kwargs.get("reg_param", 0.05),
        elasticNetParam=kwargs.get("elastic_net", 0.15), family="multinomial", tol=1e-4,
    )
    stages = [tokenizer, ngram, cv_uni, cv_bi, assembler, idf, label_indexer, clf]
    return Pipeline(stages=stages)


def map_prediction_labels(df, si_model):
    labels = list(si_model.labels)
    we = None
    for i, lbl in enumerate(labels):
        cond = F.col("pred_index").cast("int") == i
        we = F.when(cond, F.lit(lbl)) if we is None else we.when(cond, F.lit(lbl))
    return df.withColumn("pred_label", we)


spark = create_spark_session()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} siap")


## 2. Load & Train Model pada Labeled Data


In [ ]:
docs = []
with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
    cursor = client[MONGO_DB][MONGO_LABELED_COLLECTION].find(
        {TEXT_COL: {"$exists": True, "$nin": ["", None]},
         LABEL_COL: {"$exists": True, "$nin": [None, ""]}},
        {"_id": 0, "comment_id": 1, "video_id": 1, "text_original": 1, TEXT_COL: 1, LABEL_COL: 1},
    )
    for doc in cursor:
        docs.append({
            "comment_id": doc.get("comment_id", ""),
            "video_id": doc.get("video_id", ""),
            "text_original": _normalize_value(doc.get("text_original")),
            TEXT_COL: _normalize_value(doc[TEXT_COL]),
            LABEL_COL: doc.get(LABEL_COL, ""),
        })

if not docs:
    raise RuntimeError(f"Tidak ada data di {MONGO_LABELED_COLLECTION}")
train_df = spark.createDataFrame(docs).cache()
print(f"Labeled data: {train_df.count()} baris")
train_df.groupBy(LABEL_COL).count().orderBy(LABEL_COL).show()


In [ ]:
label_counts = train_df.groupBy(LABEL_COL).count().collect()
total_cnt = sum(r["count"] for r in label_counts)
n_class = len(label_counts)
weight_dict = {r[LABEL_COL]: total_cnt / (n_class * r["count"]) for r in label_counts}
print("Class weights:")
for k, v in weight_dict.items():
    print(f"  {k} -> {v:.4f}")
mapping = F.create_map(
    *[x for kv in weight_dict.items() for x in (F.lit(kv[0]), F.lit(float(kv[1])))]
)
train_df_w = train_df.withColumn("class_weight", mapping[F.col(LABEL_COL)])


In [ ]:
# Fit feature pipeline + LR dengan CrossValidator
pipeline_obj = build_pipeline()
stages = pipeline_obj.getStages()
feat_stages = stages[:-1]
feat_model = Pipeline(stages=feat_stages).fit(train_df_w)
train_feat = feat_model.transform(train_df_w).cache()

base_lr = stages[-1]
base_lr.setWeightCol("class_weight")
param_grid = ParamGridBuilder() \
    .addGrid(base_lr.regParam, [0.01, 0.05, 0.1]) \
    .addGrid(base_lr.elasticNetParam, [0.0, 0.15, 0.5]).build()
evaluator = MulticlassClassificationEvaluator(
    labelCol="label_index", predictionCol="pred_index", metricName="f1"
)
cv = CrossValidator(
    estimator=base_lr, estimatorParamMaps=param_grid,
    evaluator=evaluator, numFolds=3, seed=SEED, parallelism=4,
)
cv_model = cv.fit(train_feat)
best_lr = cv_model.bestModel
print(f"Best LR: regParam={best_lr.getRegParam():.4f}, elasticNet={best_lr.getElasticNetParam():.4f}")

# Full pipeline: feature stages + best LR
si_model = feat_model.stages[-1]
full_pipeline = Pipeline(stages=feat_stages + [best_lr])
full_model = full_pipeline.fit(train_df_w)
print("Model siap.")


## 3. Load Unlabeled Data & Predict


In [ ]:
exclude_ids = set()
with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
    for doc in client[MONGO_DB][MONGO_LABELED_COLLECTION].find(
        {}, {"comment_id": 1}
    ):
        cid = doc.get("comment_id", "")
        if cid:
            exclude_ids.add(cid)
print(f"Exclude: {len(exclude_ids)} dari {MONGO_LABELED_COLLECTION}")


In [ ]:
unlabeled_docs = []
with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
    cursor = client[MONGO_DB][MONGO_PREPROC_COLLECTION].find(
        {TEXT_COL: {"$exists": True, "$nin": ["", None]}},
        {"_id": 0, "comment_id": 1, "video_id": 1, "text_original": 1, TEXT_COL: 1},
    )
    for doc in cursor:
        cid = doc.get("comment_id", "")
        if cid in exclude_ids:
            continue
        tf = _normalize_value(doc.get(TEXT_COL))
        if not tf:
            continue
        unlabeled_docs.append({
            "comment_id": cid,
            "video_id": doc.get("video_id", ""),
            "text_original": _normalize_value(doc.get("text_original")),
            TEXT_COL: tf,
        })

print(f"Unlabeled (setelah exclude): {len(unlabeled_docs)}")
if not unlabeled_docs:
    raise RuntimeError("Tidak ada data unlabeled baru.")
unlabeled_df = spark.createDataFrame(unlabeled_docs).cache()
print(f"Unlabeled DF: {unlabeled_df.count()} rows")


In [ ]:
# Predict
unlabeled_df = unlabeled_df.withColumn(LABEL_COL, F.lit("dummy"))
pred_raw = full_model.transform(unlabeled_df).drop(LABEL_COL)
pred_df = map_prediction_labels(pred_raw, si_model)

# Ekstrak probabilitas untuk sorting
labels_in_order = list(si_model.labels)
print(f"Label order: {labels_in_order}")

extract_prob_udf = lambda idx: F.udf(lambda v: float(v[idx]), FloatType())
for lbl in labels_in_order:
    idx = labels_in_order.index(lbl)
    pred_df = pred_df.withColumn(f"prob_{lbl}", extract_prob_udf(idx)(F.col("probability")))

print("=== Distribusi Prediksi ===")
pred_df.groupBy("pred_label").count().orderBy("pred_label").show()
pred_df.select("comment_id", "video_id", "text_original", TEXT_COL, "pred_label").show(5, truncate=40)


## 4. Stratified Sampling sesuai Distribusi Target


In [ ]:
from functools import reduce

sampled_parts = []
for lbl in VALID_LABELS:
    n_target = TARGET_COUNTS[lbl]
    n_avail = pred_df.filter(F.col("pred_label") == lbl).count()
    n_take = min(n_avail, n_target)

    part = pred_df \
        .filter(F.col("pred_label") == lbl) \
        .orderBy(F.col(f"prob_{lbl}").desc()) \
        .limit(n_take) \
        .select("comment_id", "video_id", "text_original", TEXT_COL, F.col("pred_label").alias(LABEL_COL))
    sampled_parts.append(part)
    print(f"  {lbl}: target={n_target}, available={n_avail}, diambil={n_take}")

# Gabung & acak
final_df = reduce(lambda a, b: a.union(b), sampled_parts)
final_df = final_df.orderBy(F.rand(SEED))

n_total = final_df.count()
print(f"\nTotal setelah sampling: {n_total}")
final_df.groupBy(LABEL_COL).count().orderBy(LABEL_COL).show()
final_df.show(5, truncate=40)


## 5. Simpan ke MongoDB


In [ ]:
rows = final_df.collect()
al_docs = [
    {
        "comment_id": r["comment_id"],
        "video_id": r["video_id"],
        "text_original": r["text_original"],
        TEXT_COL: r[TEXT_COL],
        LABEL_COL: r[LABEL_COL],
    }
    for r in rows
]

with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
    coll = client[MONGO_DB][MONGO_OUTPUT_COLLECTION]
    coll.drop()
    if al_docs:
        coll.insert_many(al_docs, ordered=False)
        coll.create_index("comment_id", unique=True)

print(f"Disimpan ke MongoDB: {MONGO_DB}.{MONGO_OUTPUT_COLLECTION}")
print(f"Total: {len(al_docs)} dokumen")
print(f"\nDistribusi:")
from collections import Counter
dist = Counter(r[LABEL_COL] for r in al_docs)
for lbl in VALID_LABELS:
    print(f"  {lbl}: {dist.get(lbl, 0)}")


In [ ]:
spark.stop()
print("Selesai.")
